# BraTS 2021 - Lightweight 2D U-Net (single notebook)

End-to-end brain-tumor segmentation: find data -> build 2D slices -> **train / validate / test** a small 2D U-Net -> report **Dice (WT / TC / ET)** -> visualise predictions.

Built for a laptop with an **RTX 5070** and a **Miniconda** virtual environment. Every section prints or plots an output when you run it.

**BraTS labels:** 1 = necrotic core, 2 = edema, 4 = enhancing tumor (3 unused).
**Regions predicted** (3 sigmoid channels): `WT = 1,2,4`, `TC = 1,4`, `ET = 4`.

## 0. Miniconda virtual environment - full setup (do once, in a terminal)

Your **RTX 5070** is Blackwell (compute capability **sm_120**). The default PyTorch build only has kernels up to sm_90, so it crashes with *"no kernel image is available"* the moment training starts - even though `torch.cuda.is_available()` returns True. You need the **CUDA 12.8 (cu128)** build.

**Step 1 - install Miniconda** (if you don't have it): download the Windows installer from the Miniconda site and install it. You then get an app called **Anaconda Prompt (miniconda3)** in the Start menu.

**Step 2 - create and activate a virtual environment.** Open Anaconda Prompt and run:

```bat
conda create -n brats python=3.11 -y
conda activate brats
```

Your prompt now starts with `(brats)` - that means the virtual env is active.

**Step 3 - install PyTorch for the RTX 5070 (cu128) + the other libraries:**

```bat
pip install torch torchvision --index-url https://download.pytorch.org/whl/cu128
pip install nibabel opencv-python matplotlib tqdm torchinfo notebook ipykernel
```

**Step 4 - register this env as a Jupyter kernel and launch Jupyter:**

```bat
python -m ipykernel install --user --name brats --display-name "Python (brats)"
jupyter notebook
```

**Step 5** - in Jupyter, open this `.ipynb`, then top menu **Kernel -> Change kernel -> Python (brats)**. Now run the cells top to bottom.

**Everyday use later:** `conda activate brats` then `jupyter notebook`.
**When done:** `conda deactivate`. **To delete the env:** `conda env remove -n brats`.

If the GPU check (Section 3) still fails with an sm_120 error, the stable wheel is behind - swap to the nightly cu128 build inside the `(brats)` env:

```bat
pip uninstall -y torch torchvision
pip install --pre torch torchvision --index-url https://download.pytorch.org/whl/nightly/cu128 --no-cache-dir
```

*(On Kaggle none of this is needed - just turn on the GPU accelerator.)*

## 1. Imports
**Output:** installed library versions.

In [ ]:
import os, glob, random
import numpy as np
import nibabel as nib
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

try:
    from torchinfo import summary as torchinfo_summary
    HAS_TORCHINFO = True
except Exception:
    HAS_TORCHINFO = False

print("numpy   :", np.__version__)
print("torch   :", torch.__version__)
print("opencv  :", cv2.__version__)
print("nibabel :", nib.__version__)
print("torchinfo available:", HAS_TORCHINFO)

## 2. Config
Edit `DATA_ROOTS` if needed. Your settings: **batch size 16, 40 epochs, 400 patients**.

In [ ]:
# Data auto-detected from these roots (first one that has patients wins).
DATA_ROOTS = [
    r"C:\Users\Arafat\Desktop\New folder (2)\archive",  # your laptop
    "/kaggle/input",                                        # Kaggle datasets
]
OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "./brats_out"

NUM_PATIENTS = 400          # 400 of the 1251
IMG_SIZE = 128              # slices resized to 128x128
VAL_FRACTION = 0.15         # validation patients
TEST_FRACTION = 0.15        # test patients (rest = train)
KEEP_EMPTY_FRACTION = 0.10  # fraction of tumor-free slices to keep
MAX_SLICES_PER_PATIENT = 70 # cap slices/patient to bound RAM
MIN_BRAIN_FRACTION = 0.02   # skip near-black slices

EPOCHS = 40
BATCH_SIZE = 16
LR = 1e-3
BASE_FILTERS = 32           # 32 -> 7.8M params (lightweight); 16 -> ~1.9M
SEED = 42

MODALITIES = ["flair", "t1", "t1ce", "t2"]  # 4 input channels
REGIONS = ["WT", "TC", "ET"]

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
os.makedirs(OUT_DIR, exist_ok=True)

print(f"Patients={NUM_PATIENTS} | img={IMG_SIZE} | batch={BATCH_SIZE} | "
      f"epochs={EPOCHS} | base_filters={BASE_FILTERS}")
print(f"Split -> train {1-VAL_FRACTION-TEST_FRACTION:.0%} / val {VAL_FRACTION:.0%} / test {TEST_FRACTION:.0%}")
print("Output folder:", OUT_DIR)

## 3. GPU check
Runs a real matrix multiply - the only reliable test that the RTX 5070 build works.
**Output:** GPU name, compute capability, and pass/fail.

In [ ]:
def get_device():
    if not torch.cuda.is_available():
        print("[warn] CUDA not available -> CPU (slow).")
        return torch.device("cpu")
    print("GPU:", torch.cuda.get_device_name(0),
          "(sm_%d%d)" % torch.cuda.get_device_capability(0))
    try:
        _ = torch.randn(1024, 1024, device="cuda") @ torch.randn(1024, 1024, device="cuda")
        torch.cuda.synchronize()
        print("GPU compute test: OK")
    except RuntimeError as e:
        raise SystemExit("GPU compute failed - install the cu128 build (Section 0).\n" + str(e))
    return torch.device("cuda")

device = get_device()

## 4. Find the data
Searches recursively for `*_flair.nii.gz`. Unzip `BraTS2021_Training_Data.zip` first.
**Output:** how many patients were found.

In [ ]:
def find_patient_dirs(roots):
    for root in roots:
        if not os.path.isdir(root):
            continue
        flairs = glob.glob(os.path.join(root, "**", "*_flair.nii.gz"), recursive=True)
        dirs = sorted({os.path.dirname(f) for f in flairs})
        if dirs:
            print(f"Found {len(dirs)} patients under {root}")
            return dirs
    raise SystemExit("No patients found. Unzip the data or edit DATA_ROOTS.")

patients = find_patient_dirs(DATA_ROOTS)
print("Example patient folder:", os.path.basename(patients[0]))

## 5. Preprocessing functions
Z-score normalise each modality on brain voxels, keep tumor slices (+ a few empty), resize to 128x128, build WT/TC/ET mask channels.
**Output:** shapes for one test patient.

In [ ]:
def znorm(vol):
    brain = vol > 0
    if brain.sum() == 0:
        return vol
    vol = (vol - vol[brain].mean()) / (vol[brain].std() + 1e-8)
    vol[~brain] = 0.0
    return vol

def load_vol(pdir, modality):
    pid = os.path.basename(pdir)
    return nib.load(os.path.join(pdir, f"{pid}_{modality}.nii.gz")).get_fdata().astype(np.float32)

def rz_img(sl):
    return cv2.resize(sl, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)

def rz_msk(sl):
    return cv2.resize(sl.astype(np.uint8), (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)

def build_patient_slices(pdir):
    mods = [znorm(load_vol(pdir, m)) for m in MODALITIES]
    seg = load_vol(pdir, "seg")
    Z = seg.shape[2]
    keep = []
    for z in range(Z):
        if (mods[0][:, :, z] != 0).mean() < MIN_BRAIN_FRACTION:
            continue
        if seg[:, :, z].max() > 0 or random.random() < KEEP_EMPTY_FRACTION:
            keep.append(z)
    if len(keep) > MAX_SLICES_PER_PATIENT:
        idx = np.linspace(0, len(keep) - 1, MAX_SLICES_PER_PATIENT).astype(int)
        keep = [keep[i] for i in idx]
    imgs, masks = [], []
    for z in keep:
        img = np.stack([rz_img(m[:, :, z]) for m in mods], 0).astype(np.float16)
        s = seg[:, :, z]
        mask = np.stack([rz_msk(np.isin(s, [1, 2, 4])),
                         rz_msk(np.isin(s, [1, 4])),
                         rz_msk(s == 4)], 0).astype(np.uint8)
        imgs.append(img); masks.append(mask)
    if not imgs:
        return None, None
    return np.stack(imgs), np.stack(masks)

_img, _msk = build_patient_slices(patients[0])
print("One patient -> images", _img.shape, "| masks", _msk.shape)
print("(N_slices, 4 modalities, H, W) and (N_slices, 3 regions, H, W)")

## 6. Build train / val / test arrays (split by patient)
Splitting by patient - never by slice - keeps slices from one patient out of multiple sets, so the Dice scores are honest.
**Output:** slice counts per split and RAM used.

In [ ]:
def build_dataset(patient_dirs):
    patient_dirs = list(patient_dirs)
    random.shuffle(patient_dirs)
    patient_dirs = patient_dirs[:NUM_PATIENTS]
    n_val = int(len(patient_dirs) * VAL_FRACTION)
    n_test = int(len(patient_dirs) * TEST_FRACTION)
    val_set = set(patient_dirs[:n_val])
    test_set = set(patient_dirs[n_val:n_val + n_test])

    buckets = {"train": [[], []], "val": [[], []], "test": [[], []]}
    for pdir in tqdm(patient_dirs, desc="Slicing patients"):
        imgs, masks = build_patient_slices(pdir)
        if imgs is None:
            continue
        split = "val" if pdir in val_set else "test" if pdir in test_set else "train"
        buckets[split][0].append(imgs)
        buckets[split][1].append(masks)

    out = {}
    for k, (xs, ys) in buckets.items():
        out[k] = (np.concatenate(xs), np.concatenate(ys))
    return out

data = build_dataset(patients)
(Xtr, Ytr), (Xva, Yva), (Xte, Yte) = data["train"], data["val"], data["test"]
ram = (Xtr.nbytes + Xva.nbytes + Xte.nbytes) / 1e9
print(f"Train slices: {len(Xtr):5d}")
print(f"Val   slices: {len(Xva):5d}")
print(f"Test  slices: {len(Xte):5d}")
print(f"Images in RAM: ~{ram:.1f} GB")

## 6b. Peek at one preprocessed slice
**Output:** the 4 modalities and the tumor mask for a validation slice with tumor.

In [ ]:
def show_sample(X, Y):
    idx = next((i for i in range(len(Y)) if Y[i, 0].sum() > 50), 0)
    fig, ax = plt.subplots(1, 5, figsize=(15, 3))
    for j, name in enumerate(MODALITIES):
        ax[j].imshow(X[idx, j].astype(np.float32), cmap="gray")
        ax[j].set_title(name); ax[j].axis("off")
    ax[4].imshow(X[idx, 0].astype(np.float32), cmap="gray")
    ax[4].imshow(Y[idx, 0], alpha=0.4, cmap="Reds")   # Whole Tumor overlay
    ax[4].set_title("WT mask"); ax[4].axis("off")
    plt.tight_layout(); plt.show()

show_sample(Xva, Yva)

## 7. Dataset + DataLoaders
**Output:** shape of one training batch.

In [ ]:
class SliceDS(Dataset):
    def __init__(self, X, Y):
        self.X, self.Y = X, Y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, i):
        return (torch.from_numpy(self.X[i].astype(np.float32)),
                torch.from_numpy(self.Y[i].astype(np.float32)))

pin = device.type == "cuda"
train_loader = DataLoader(SliceDS(Xtr, Ytr), batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=pin)
val_loader   = DataLoader(SliceDS(Xva, Yva), batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=pin)
test_loader  = DataLoader(SliceDS(Xte, Yte), batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=pin)

xb, yb = next(iter(train_loader))
print("Batch images:", tuple(xb.shape), "| Batch masks:", tuple(yb.shape))

## 8. The 2D U-Net + its internal architecture
**Output:** the full module tree, a layer-by-layer summary with output shapes, and total parameters.

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(True),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(True))
    def forward(self, x):
        return self.net(x)

class UNet2D(nn.Module):
    def __init__(self, in_ch=4, out_ch=3, base=32):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.d1 = DoubleConv(in_ch, base)
        self.d2 = DoubleConv(base, base*2)
        self.d3 = DoubleConv(base*2, base*4)
        self.d4 = DoubleConv(base*4, base*8)
        self.bott = DoubleConv(base*8, base*16)
        self.up4 = nn.ConvTranspose2d(base*16, base*8, 2, stride=2); self.u4 = DoubleConv(base*16, base*8)
        self.up3 = nn.ConvTranspose2d(base*8, base*4, 2, stride=2);  self.u3 = DoubleConv(base*8, base*4)
        self.up2 = nn.ConvTranspose2d(base*4, base*2, 2, stride=2);  self.u2 = DoubleConv(base*4, base*2)
        self.up1 = nn.ConvTranspose2d(base*2, base, 2, stride=2);    self.u1 = DoubleConv(base*2, base)
        self.out = nn.Conv2d(base, out_ch, 1)
    def forward(self, x):
        c1 = self.d1(x); c2 = self.d2(self.pool(c1))
        c3 = self.d3(self.pool(c2)); c4 = self.d4(self.pool(c3))
        b = self.bott(self.pool(c4))
        x = self.u4(torch.cat([self.up4(b), c4], 1))
        x = self.u3(torch.cat([self.up3(x), c3], 1))
        x = self.u2(torch.cat([self.up2(x), c2], 1))
        x = self.u1(torch.cat([self.up1(x), c1], 1))
        return self.out(x)

model = UNet2D(in_ch=4, out_ch=3, base=BASE_FILTERS).to(device)

print("================ U-Net module tree ================")
print(model)
n_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {n_params/1e6:.2f}M")

print("\n============ Layer-by-layer summary ============")
if HAS_TORCHINFO:
    torchinfo_summary(model, input_size=(1, 4, IMG_SIZE, IMG_SIZE),
                      col_names=("input_size", "output_size", "num_params"), depth=3)
else:
    print("torchinfo not installed - showing per-block output shapes instead:")
    x = torch.randn(1, 4, IMG_SIZE, IMG_SIZE, device=device)
    with torch.no_grad():
        c1 = model.d1(x); print(" enc1 :", tuple(c1.shape))
        c2 = model.d2(model.pool(c1)); print(" enc2 :", tuple(c2.shape))
        c3 = model.d3(model.pool(c2)); print(" enc3 :", tuple(c3.shape))
        c4 = model.d4(model.pool(c3)); print(" enc4 :", tuple(c4.shape))
        b = model.bott(model.pool(c4)); print(" bott :", tuple(b.shape))
        print(" out  :", tuple(model(x).shape))

## 9. Loss (Dice + BCE) and the Dice metric
**Output:** a sanity check - a perfect prediction must give Dice = 1.0.

In [ ]:
class DiceBCELoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
    def forward(self, logits, y, eps=1e-6):
        probs = torch.sigmoid(logits)
        inter = (probs * y).sum((0, 2, 3))
        denom = probs.sum((0, 2, 3)) + y.sum((0, 2, 3))
        dice = (2 * inter + eps) / (denom + eps)
        return self.bce(logits, y) + (1 - dice.mean())

@torch.no_grad()
def accumulate_dice(logits, y, inter, denom, thr=0.5):
    preds = (torch.sigmoid(logits) > thr).float()
    inter += (preds * y).sum((0, 2, 3)).cpu()
    denom += (preds.sum((0, 2, 3)) + y.sum((0, 2, 3))).cpu()
    return inter, denom

def finalize_dice(inter, denom, eps=1e-6):
    return (2 * inter + eps) / (denom + eps)

# sanity check
_y = (torch.rand(2, 3, 32, 32) > 0.5).float()
_perfect = (_y - 0.5) * 20.0
_i, _d = accumulate_dice(_perfect, _y, torch.zeros(3), torch.zeros(3))
print("Dice on a perfect prediction:", [round(v, 3) for v in finalize_dice(_i, _d).tolist()], "(expect ~1.0)")

## 10. Train + validate (40 epochs)
Mixed precision (fast on the RTX 5070). **Output:** per-epoch **train loss, val loss, and val Dice** progress, plus training curves. Best model saved to `best_model.pt`.

In [ ]:
def train_model(model, train_loader, val_loader, device):
    torch.backends.cudnn.benchmark = True
    loss_fn = DiceBCELoss()
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=amp)

    hist = {k: [] for k in ["train", "val", "WT", "TC", "ET"]}
    best = -1.0
    for ep in range(1, EPOCHS + 1):
        model.train(); run = 0.0
        for x, y in tqdm(train_loader, leave=False, desc=f"epoch {ep}/{EPOCHS} [train]"):
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with torch.amp.autocast("cuda", enabled=amp):
                loss = loss_fn(model(x), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            run += loss.item() * x.size(0)
        tr_loss = run / len(train_loader.dataset)

        model.eval(); run = 0.0
        inter, denom = torch.zeros(3), torch.zeros(3)
        with torch.no_grad():
            for x, y in tqdm(val_loader, leave=False, desc=f"epoch {ep}/{EPOCHS} [val]"):
                x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
                with torch.amp.autocast("cuda", enabled=amp):
                    logits = model(x); loss = loss_fn(logits, y)
                run += loss.item() * x.size(0)
                inter, denom = accumulate_dice(logits.float(), y, inter, denom)
        va_loss = run / len(val_loader.dataset)
        wt, tc, et = finalize_dice(inter, denom).tolist()
        mean = (wt + tc + et) / 3
        sched.step()

        for k, v in zip(hist, [tr_loss, va_loss, wt, tc, et]):
            hist[k].append(v)
        star = ""
        if mean > best:
            best = mean
            torch.save({"model": model.state_dict(), "base": BASE_FILTERS,
                        "epoch": ep, "dice": {"WT": wt, "TC": tc, "ET": et}},
                       os.path.join(OUT_DIR, "best_model.pt"))
            star = "  <- best saved"
        print(f"Epoch {ep:02d}/{EPOCHS} | train {tr_loss:.4f} | val {va_loss:.4f} | "
              f"val Dice  WT {wt:.3f}  TC {tc:.3f}  ET {et:.3f} | mean {mean:.3f}{star}")

    ep_range = range(1, EPOCHS + 1)
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    ax[0].plot(ep_range, hist["train"], label="train loss")
    ax[0].plot(ep_range, hist["val"], label="val loss")
    ax[0].set_title("Loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
    for r in REGIONS:
        ax[1].plot(ep_range, hist[r], label=f"val {r}")
    ax[1].set_title("Validation Dice"); ax[1].set_ylim(0, 1); ax[1].set_xlabel("epoch"); ax[1].legend()
    fig.tight_layout(); fig.savefig(os.path.join(OUT_DIR, "training_curves.png"), dpi=120); plt.show()
    print(f"\nBest mean validation Dice: {best:.3f}")
    return hist

history = train_model(model, train_loader, val_loader, device)

## 11. Test evaluation (held-out patients)
Loads the best saved model and scores it on the **test** set - patients never seen in training or validation.
**Output:** test Dice for WT, TC, ET.

In [ ]:
ckpt = torch.load(os.path.join(OUT_DIR, "best_model.pt"), map_location=device)
best_model = UNet2D(4, 3, ckpt["base"]).to(device)
best_model.load_state_dict(ckpt["model"]); best_model.eval()
print(f"Loaded best model from epoch {ckpt['epoch']}")

inter, denom = torch.zeros(3), torch.zeros(3)
with torch.no_grad():
    for x, y in tqdm(test_loader, desc="Testing"):
        x, y = x.to(device), y.to(device)
        with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
            logits = best_model(x)
        inter, denom = accumulate_dice(logits.float(), y, inter, denom)
wt, tc, et = finalize_dice(inter, denom).tolist()
print("\n================ TEST Dice ================")
print(f"  WT (Whole Tumor)     : {wt:.3f}")
print(f"  TC (Tumor Core)      : {tc:.3f}")
print(f"  ET (Enhancing Tumor) : {et:.3f}")
print(f"  Mean                 : {(wt+tc+et)/3:.3f}")

## 12. Visualise test predictions
**Output:** FLAIR / ground-truth / prediction for a few test slices, with per-slice Dice.

In [ ]:
def visualize(model, X, Y, device, n_show=6):
    model.eval()
    tumor_idx = [i for i in range(len(Y)) if Y[i, 0].sum() > 50]
    picks = random.sample(tumor_idx, min(n_show, len(tumor_idx)))
    fig, axes = plt.subplots(len(picks), 3, figsize=(9, 3 * len(picks)))
    if len(picks) == 1:
        axes = axes[None, :]
    for r, i in enumerate(picks):
        x = torch.from_numpy(X[i].astype(np.float32)).unsqueeze(0).to(device)
        with torch.no_grad(), torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
            pred = (torch.sigmoid(model(x))[0, 0] > 0.5).cpu().numpy()
        gt = Y[i, 0]; flair = X[i, 0].astype(np.float32)
        d = (2 * (pred * gt).sum() + 1e-6) / (pred.sum() + gt.sum() + 1e-6)
        axes[r, 0].imshow(flair, cmap="gray"); axes[r, 0].set_title("FLAIR")
        axes[r, 1].imshow(flair, cmap="gray"); axes[r, 1].imshow(gt, alpha=0.4, cmap="Reds"); axes[r, 1].set_title("Ground truth (WT)")
        axes[r, 2].imshow(flair, cmap="gray"); axes[r, 2].imshow(pred, alpha=0.4, cmap="Reds"); axes[r, 2].set_title(f"Prediction (Dice {d:.2f})")
        for a in axes[r]:
            a.axis("off")
    fig.tight_layout(); fig.savefig(os.path.join(OUT_DIR, "predictions.png"), dpi=120); plt.show()

visualize(best_model, Xte, Yte, device)

## Notes on reading the Dice scores
- **WT** (Whole Tumor) is easiest - expect the highest number.
- **TC** (Tumor Core) is mid.
- **ET** (Enhancing Tumor) is hardest and noisiest; a lower ET Dice is normal for a 2D model on a patient subset.
- A 2D slice model won't match full 3D BraTS leaderboards - this is a fast, understandable baseline.

**Lighter/faster:** `BASE_FILTERS = 16`, fewer `NUM_PATIENTS`, lower `MAX_SLICES_PER_PATIENT`.
**Stronger:** more `EPOCHS`, `IMG_SIZE = 160`, `BASE_FILTERS = 48`, more patients.
**Out of GPU memory:** lower `BATCH_SIZE` to 8.